In financial services and markets, instruments and securities have unique identifiers. These identifiers are essential to ensure that all parties involved in trading are referring to the same instrument. For example, equity securities may be traded on different exchanges in the USA and globally. Products such as bonds, currencies, and structured derivatives each have their own unique identifiers and conventions for creating new ones. Common identifiers include Ticker, ISIN, CUSIP, SEDOL, RIC, BBGID, and others.

In [ ]:
#import necessary libraries
import requests
import json
import pandas as pd
import pprint
import os
import dotenv
from dotenv import load_dotenv
load_dotenv()


True

In [ ]:
#get identifiers from DAX. DAX listed companies only have ISIN and WKN identifiers. 
# Random list of 5 tickers on DAX for testing purposes

ticks = ['ADS', 'BMW', 'FME', 'IFX', 'HEI'] #, 'DPW.DE', 'DTE.DE', 'FME.DE', 'FRE.DE', 'HEI.DE', 'HEN3.DE', 'IFX.DE', 'LIN.DE', 'MRK.DE', 'MUV2.DE', 'RWE.DE', 'SAP.DE', 'SIE.DE', 'VOW3.DE']

In [ ]:
#Market Identifier Code(mic) for Xetra Frankfurt is 'XETR'
jobs = [{'idType': 'TICKER','idValue': tick, 'micCode': 'XETR'} for tick in ticks]



In [130]:
openfigi_url = 'https://api.openfigi.com/v3/mapping'
headers = {'Content-Type': 'application/json'}
response = requests.post(openfigi_url, headers=headers, data=json.dumps(jobs))
if response.status_code != 200:
    raise Exception(f"Error: {response.status_code}, {response.text}")
result = response.json()

In [131]:
pprint.pprint(result)

[{'data': [{'compositeFIGI': 'BBG000FR1Q22',
            'exchCode': 'GY',
            'figi': 'BBG000FR1RP5',
            'marketSector': 'Equity',
            'name': 'ADIDAS AG',
            'securityDescription': 'ADS',
            'securityType': 'Common Stock',
            'securityType2': 'Common Stock',
            'shareClassFIGI': 'BBG001S8J8Q3',
            'ticker': 'ADS'}]},
 {'data': [{'compositeFIGI': 'BBG000BBX8Q0',
            'exchCode': 'GY',
            'figi': 'BBG000BBXB74',
            'marketSector': 'Equity',
            'name': 'BAYERISCHE MOTOREN WERKE AG',
            'securityDescription': 'BMW',
            'securityType': 'Common Stock',
            'securityType2': 'Common Stock',
            'shareClassFIGI': 'BBG001S676R3',
            'ticker': 'BMW'}]},
 {'data': [{'compositeFIGI': 'BBG000DHXQT2',
            'exchCode': 'GY',
            'figi': 'BBG000DHXTF1',
            'marketSector': 'Equity',
            'name': 'FRESENIUS MEDICAL CARE AG',
  

In [132]:
#just to show the dataframe manipulation steps # type: ignore
just_dict = [d['data'][0] for d in result]
df_figi = pd.DataFrame(just_dict)
df_figi

,figi,name,ticker,exchCode,compositeFIGI,securityType,marketSector,shareClassFIGI,securityType2,securityDescription
0,BBG000FR1RP5,ADIDAS AG,ADS,GY,BBG000FR1Q22,Common Stock,Equity,BBG001S8J8Q3,Common Stock,ADS
1,BBG000BBXB74,BAYERISCHE MOTOREN WERKE AG,BMW,GY,BBG000BBX8Q0,Common Stock,Equity,BBG001S676R3,Common Stock,BMW
2,BBG000DHXTF1,FRESENIUS MEDICAL CARE AG,FME,GY,BBG000DHXQT2,Common Stock,Equity,BBG001S88634,Common Stock,FME
3,BBG000C8L273,INFINEON TECHNOLOGIES AG,IFX,GY,BBG000C8L0S4,Common Stock,Equity,BBG001S72847,Common Stock,IFX
4,BBG000BC1KH6,HEIDELBERG MATERIALS AG,HEI,GY,BBG000BC1HT0,Common Stock,Equity,BBG001S68KS0,Common Stock,HEI


In [133]:
#columns of interest and set ticker as index
columns_of_interest = ['name', 'ticker', 'marketSector', 'exchCode', 'figi']
df_figi = df_figi[columns_of_interest].set_index('ticker')
df_figi

,name,marketSector,exchCode,figi
ticker,,,,
ADS,ADIDAS AG,Equity,GY,BBG000FR1RP5
BMW,BAYERISCHE MOTOREN WERKE AG,Equity,GY,BBG000BBXB74
FME,FRESENIUS MEDICAL CARE AG,Equity,GY,BBG000DHXTF1
IFX,INFINEON TECHNOLOGIES AG,Equity,GY,BBG000C8L273
HEI,HEIDELBERG MATERIALS AG,Equity,GY,BBG000BC1KH6


Refinitv PermID section

In [134]:
access_token = os.getenv('API_KEY3') # get API key from environment variable

In [135]:
#API endpoint for Refinitiv PermID
url = "https://api-eit.refinitiv.com/permid/match"
headers = {
    'Content-Type': 'text/plain',
    'Accept': 'application/json',
    'X-AG-Access-Token': access_token,
    'x-openmatch-numberOfMatchesPerRecord': '1',
    'x-openmatch-dataType': 'Organization',
}

In [136]:
#the first line in the etxt field is the standard identifier. We use 'Ticker'
text_field = 'Standard Identifier\n'

for tick in ticks:
    text_field += f'TICKER:{tick}' +'&&MIC:' + 'XETR\n'
print(text_field)

Standard Identifier
TICKER:ADS&&MIC:XETR
TICKER:BMW&&MIC:XETR
TICKER:FME&&MIC:XETR
TICKER:IFX&&MIC:XETR
TICKER:HEI&&MIC:XETR



In [137]:
response = requests.post(url, headers=headers, data=text_field)
r = response.json()
print(response.headers)
#print(r)

{'Date': 'Tue, 21 Oct 2025 09:44:49 GMT', 'Content-Type': 'application/json; charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'X-RateLimit-Limit-day': '105000', 'RateLimit-Remaining': '104994', 'RateLimit-Limit': '105000', 'X-RateLimit-Remaining-day': '104994', 'RateLimit-Reset': '51311', 'X-RateLimit-Remaining-RequestSize': '499999875', 'X-RateLimit-Limit-RequestSize': '500000000', 'Set-Cookie': 'TR-TMS-MDaaS-SESSIONID=AZoGKDVMsBTjkc1DJoHT^5ae3d8c4;Path=/;HttpOnly', 'Expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'vary': 'Origin', 'Access-Control-Allow-Credentials': 'true', 'Strict-Transport-Security': 'max-age=31536000'}


In [138]:
pprint.pprint(r['outputContentResponse'])

[{'Input_Standard Identifier': 'TICKER:ADS&&MIC:XETR',
  'Match Level': 'Excellent',
  'Match OpenPermID': 'https://permid.org/1-4295868725',
  'Match Ordinal': '1',
  'Match OrgName': 'Adidas AG',
  'Match Score': '100%',
  'Original Row Number': '2',
  'ProcessingStatus': 'OK'},
 {'Input_Standard Identifier': 'TICKER:BMW&&MIC:XETR',
  'Match Level': 'Excellent',
  'Match OpenPermID': 'https://permid.org/1-4295869227',
  'Match Ordinal': '1',
  'Match OrgName': 'Bayerische Motoren Werke AG',
  'Match Score': '100%',
  'Original Row Number': '3',
  'ProcessingStatus': 'OK'},
 {'Input_Standard Identifier': 'TICKER:FME&&MIC:XETR',
  'Match Level': 'Excellent',
  'Match OpenPermID': 'https://permid.org/1-4295869203',
  'Match Ordinal': '1',
  'Match OrgName': 'Fresenius Medical Care AG',
  'Match Score': '100%',
  'Original Row Number': '4',
  'ProcessingStatus': 'OK'},
 {'Input_Standard Identifier': 'TICKER:IFX&&MIC:XETR',
  'Match Level': 'Excellent',
  'Match OpenPermID': 'https://perm

In [139]:
for company in r['outputContentResponse']:
    print(company['Match OrgName'] + ' --> ' + company['Match OpenPermID'])

Adidas AG --> https://permid.org/1-4295868725
Bayerische Motoren Werke AG --> https://permid.org/1-4295869227
Fresenius Medical Care AG --> https://permid.org/1-4295869203
Infineon Technologies AG --> https://permid.org/1-4295870063
Heidelberg Materials AG --> https://permid.org/1-4295868961


In [140]:
#define a function to retrieve data from the urls
def permid_data(permid_url):
    permid_headers = {
        'Accept': 'text/turtle'
    }

    permid_params = {
        'format': 'json-ld',
        'access-token': access_token
    }

    #actual request
    permid_response = requests.get(permid_url, headers=headers,params=permid_params)

    #convrt the response to JSON
    permid_data = json.loads(permid_response.content)

    return permid_data

In [141]:
#create empty dictionary
permid_dict = {}

#loop thru all tickers and put the data in dictionary

for tick, i in zip(ticks, r['outputContentResponse']):
    #the PermidID url for the ticker from the reponse earlier
    permid_url = i['Match OpenPermID']

    #'use function efined above to download data'
    data = permid_data(permid_url)

    #Put desired data in dictionary for the ticker
    permid_dict[tick] = {
        'company': data['vcard:organization-name'],
        'IPO'    : data['hasIPODate'],
        'address': data['mdaas:HeadquartersAddress'],
        'website': data['hasURL'],
        'phone'  : data['tr-org:hasHeadquartersPhoneNumber'],
        'LEI'    : data['tr-org:hasLEI'],
        'permid' : data['tr-common:hasPermId'],
        'permid_url' : permid_url
    }

In [142]:
df_permid = pd.DataFrame.from_dict(permid_dict, orient='index')

df_permid

,company,IPO,address,website,phone,LEI,permid,permid_url
ADS,Adidas AG,1997-11-28T05:00:00Z,Adi-Dassler-Strasse 1\nHERZOGENAURACH\nBAYERN\...,https://www.adidas-group.com/,499132842352,549300JSX0Z4CW0V5023,4295868725,https://permid.org/1-4295868725
BMW,Bayerische Motoren Werke AG,1926-01-01T05:00:00Z,Petuelring 130\nMUENCHEN\nBAYERN\n80809\nGerma...,https://www.bmwgroup.com/,49893820,YEH5ZCD6E441RHVHD759,4295869227,https://permid.org/1-4295869227
FME,Fresenius Medical Care AG,1996-10-02T04:00:00Z,Else-Kroener-Strasse 1\nBAD HOMBURG VOR DER HO...,https://www.freseniusmedicalcare.com/,4961726090,549300CP8NY40UP89Q40,4295869203,https://permid.org/1-4295869203
IFX,Infineon Technologies AG,2000-03-13T05:00:00Z,Am Campeon 1-12\nMUNICH\nBAYERN\n85579\nGermany\n,https://www.infineon.com/,491149892340,TSI2PJM6EPETEQ4X1U25,4295870063,https://permid.org/1-4295870063
HEI,Heidelberg Materials AG,1997-11-28T05:00:00Z,Berliner Strasse 6\nHEIDELBERG\nBADEN-WUERTTEM...,https://www.heidelbergmaterials.com/,4962214810,LZ2C6E0W5W7LQMX5ZI37,4295868961,https://permid.org/1-4295868961


In [143]:
#df_final=df_figi.join(df_permid)

#df_final

In [144]:
import requests

url = "https://api.marginalia.nu/{key}/search/{query}";

rsp = requests.get(url.format(key='public', query="Lake Vostok"));

if rsp.ok:
  data = rsp.json()
  print ("Query: ", data['query'])
  print ("License: ", data['license'])
  print ("")
  for result in data['results']:
      print (result['url'])
      print ("\t" + result['title'])
      print ("\t" + result['description'])
      print ("")
else:
    print ("Bad Status " + str(rsp.status_code))

Query:  Lake Vostok
License:  CC-BY-NC-SA 4.0

https://en.wikipedia.org/wiki/Lake_Vostok
	Lake Vostok
	Lake Vostok (Russian romanized: ozero Vostok is the largest of Antarctica's 675 known subglacial lakes. Lake Vostok is located at the southern Pole of Cold, beneath Russia's Vostok Station under the surface of the central East Antarctic Ice Sheet, which i

https://southpolestation.com/trivia/10s/lakevostok.html
	Lake Vostok Drilling Project
	There wasn't any news. Apparently Russia did not conduct any drilling-related activity at Vostok in 2015-16 the only thing they presented at the 2016 Antarctic Treaty meeting in Santiago, Chile was a report containing the results of laboratory experiments

http://www.greatdreams.com/blog-2012/dee-blog130.html
	Dee Finney's blog  February 7, 2012  page 130  WHAT'S UP 
WITH LAKE VOSTOK?
	I deal with interpreteting aeromagnetic imagery daily in my mineral exploration work here in OZ. The huge size and intensity of the above mentioned magnetic anomaly